In [6]:
import anndata as ad
import numpy as np
import pandas as pd
from scipy import stats

# ── 1. Load / point to your AnnData ──────────────────────────────────────────
adata = ad.read_h5ad("/home/workspace/private/working/reducedMarkers_R05clustered_adata_032426.h5ad")   # ← update path

# ── 2. Define DC marker panels ────────────────────────────────────────────────
DC_POSITIVE = [
    'CD1c-OTI2F4AF750',
    'CD11c-D3V1EAF647',
    'CD452B11PD726AF750', 
]
DC_NEGATIVE = [
    'CD14-D7A2TAF488',
]

LAYER = 'normalized'   # use the normalised layer for fair comparison

# ── 3. Validate markers are present ──────────────────────────────────────────
all_markers = DC_POSITIVE + DC_NEGATIVE
missing = [m for m in all_markers if m not in adata.var_names]
if missing:
    print(f"⚠ Markers not found in adata.var — check spelling:\n  {missing}")
    # fuzzy-match helper to suggest alternatives
    for m in missing:
        close = [v for v in adata.var_names if any(part in v for part in m.split('-'))]
        if close:
            print(f"  Possible matches for '{m}': {close}")

present_pos = [m for m in DC_POSITIVE if m not in missing]
present_neg = [m for m in DC_NEGATIVE if m not in missing]

# ── 4. Extract expression matrix for relevant markers ────────────────────────
import scipy.sparse as sp

def get_expr(adata, markers, layer):
    """Return a dense DataFrame (cells × markers) from the chosen layer."""
    idx = [adata.var_names.get_loc(m) for m in markers]
    X = adata.layers[layer]
    sub = X[:, idx]
    if sp.issparse(sub):
        sub = sub.toarray()
    return pd.DataFrame(sub, index=adata.obs_names, columns=markers)

expr_pos = get_expr(adata, present_pos, LAYER) if present_pos else pd.DataFrame()
expr_neg = get_expr(adata, present_neg, LAYER) if present_neg else pd.DataFrame()

leiden = adata.obs['leiden'].values

# ── 5. Per-cluster statistics ─────────────────────────────────────────────────
clusters = sorted(adata.obs['leiden'].unique(), key=lambda x: int(x))

rows = []
for cl in clusters:
    mask = leiden == cl
    row = {'leiden': cl, 'n_cells': mask.sum()}

    # mean expression per marker
    for m in present_pos:
        row[f'mean_{m}'] = expr_pos.loc[mask, m].mean()
    for m in present_neg:
        row[f'mean_{m}'] = expr_neg.loc[mask, m].mean()

    rows.append(row)

df = pd.DataFrame(rows).set_index('leiden')

# ── 6. Z-score each marker across clusters (so they're on the same scale) ────
mean_pos_cols = [f'mean_{m}' for m in present_pos]
mean_neg_cols = [f'mean_{m}' for m in present_neg]

for col in mean_pos_cols + mean_neg_cols:
    df[f'z_{col}'] = stats.zscore(df[col].astype(float))

# ── 7. Composite DC score ─────────────────────────────────────────────────────
#   + average z-score of positive markers   (high = good)
#   − average z-score of negative markers   (high = bad)
pos_z_cols = [f'z_mean_{m}' for m in present_pos]
neg_z_cols = [f'z_mean_{m}' for m in present_neg]

df['score_positive'] = df[pos_z_cols].mean(axis=1) if pos_z_cols else 0
df['score_negative'] = df[neg_z_cols].mean(axis=1) if neg_z_cols else 0
df['dc_composite_score'] = df['score_positive'] - df['score_negative']

# ── 8. Rank and display ───────────────────────────────────────────────────────
df_ranked = df.sort_values('dc_composite_score', ascending=False)

print("\n── DC Cluster Ranking ───────────────────────────────────────────────────")
display_cols = (
    ['n_cells']
    + [f'mean_{m}' for m in present_pos + present_neg]
    + ['score_positive', 'score_negative', 'dc_composite_score']
)
print(df_ranked[display_cols].to_string(float_format='{:.4f}'.format))

# ── 9. Optional: save to CSV ─────────────────────────────────────────────────
df_ranked[display_cols].to_csv("dc_cluster_ranking.csv")
print("\n✓ Saved → dc_cluster_ranking.csv")


── DC Cluster Ranking ───────────────────────────────────────────────────
        n_cells  mean_CD1c-OTI2F4AF750  mean_CD11c-D3V1EAF647  mean_CD452B11PD726AF750  mean_CD14-D7A2TAF488  score_positive  score_negative  dc_composite_score
leiden                                                                                                                                                          
17        41275               520.4863               174.8203                 421.2233              126.5383          1.9722         -0.4509              2.4230
7        257768               176.9355               267.5357                1057.9833              164.0192          1.6854          0.0314              1.6540
15       148703               175.1819               110.3512                 176.7313               43.2264         -0.4871         -1.5229              1.0358
16        89021               154.5646               109.0895                 166.3470               36.9617         -0.